# Continuous Lunar Lander with Soft Actor-Critic

`LunarLander-v3` is discrete by default. Here we deliberately request
`continuous=True`, giving SAC a two-dimensional bounded action: main-engine
throttle and lateral-engine throttle. This is the right version for a
squashed-Gaussian actor; DQN would be the natural baseline for the default
four-action version.

The notebook has two execution profiles. `QUICK=True` checks the complete
training/evaluation/animation workflow on CPU but is **not a learning claim**.
Set `QUICK=False` for the preregistered 500k-step profile. Automated tests set
`RL_LAB_NOTEBOOK_SMOKE=1` for an even smaller structural run.


In [ ]:
from __future__ import annotations

import copy
from dataclasses import asdict, dataclass
import os
from pathlib import Path
import random
import time

import gymnasium as gym
from gymnasium.spaces import Box
from IPython.display import HTML, display
from matplotlib import animation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.distributions import Normal

SEED = 23
QUICK = True
SMOKE = os.environ.get("RL_LAB_NOTEBOOK_SMOKE") == "1"
DEVICE = torch.device("cpu")  # Small MLPs are reproducible and fast on CPU.

available_styles = set(plt.style.available)
plot_style = next(
    (
        style
        for style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot")
        if style in available_styles
    ),
    "default",
)
plt.style.use(plot_style)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


@dataclass(frozen=True)
class SACConfig:
    env_id: str = "LunarLander-v3"
    seed: int = SEED
    total_steps: int = 320 if SMOKE else (8_000 if QUICK else 500_000)
    learning_starts: int = 64 if SMOKE else (1_000 if QUICK else 10_000)
    replay_capacity: int = 2_000 if SMOKE else (100_000 if QUICK else 1_000_000)
    batch_size: int = 32 if SMOKE else 256
    hidden_sizes: tuple[int, ...] = (
        (32, 32) if SMOKE else ((128, 128) if QUICK else (400, 300))
    )
    actor_lr: float = 3e-4
    critic_lr: float = 3e-4
    alpha_lr: float = 3e-4
    gamma: float = 0.99
    tau: float = 0.01
    initial_alpha: float = 0.2
    max_grad_norm: float = 10.0
    eval_interval: int = 160 if SMOKE else (2_000 if QUICK else 10_000)
    eval_episodes: int = 1 if SMOKE else (3 if QUICK else 10)
    diagnostic_interval: int = 8 if SMOKE else (25 if QUICK else 100)


config = SACConfig()
seed_everything(config.seed)

probe = gym.make(config.env_id, continuous=True, enable_wind=False)
try:
    observation, _ = probe.reset(seed=config.seed)
    probe.action_space.seed(config.seed)
    assert isinstance(probe.action_space, Box)
    assert observation.shape == (8,) and probe.action_space.shape == (2,)
    OBS_DIM = int(np.prod(probe.observation_space.shape))
    ACT_DIM = int(np.prod(probe.action_space.shape))
    ACTION_LOW = probe.action_space.low.astype(np.float32).copy()
    ACTION_HIGH = probe.action_space.high.astype(np.float32).copy()
finally:
    probe.close()

print(f"device={DEVICE}; observation={OBS_DIM}; action={ACT_DIM}")
display(pd.Series(asdict(config), name="value").to_frame())


## 1. Environment contract and SAC objectives

The state is $(x,y,v_x,v_y,\theta,\dot\theta,c_L,c_R)$. For continuous
actions $a=(a_{main},a_{lateral})\in[-1,1]^2$, the main engine is off below
zero and each lateral engine has a dead zone $|a_{lateral}|<0.5$. We therefore
retain engine-activation diagnostics rather than treating the action vector
as an opaque control.

SAC learns two critics and uses the smaller target:

$$y=r+\gamma(1-z)\left[\min_i Q_{\bar\phi_i}(s',a')
  -\alpha\log\pi_\theta(a'\mid s')\right],\qquad
  a'\sim\pi_\theta(\cdot\mid s'),$$

$$J_{Q_i}=\mathbb E[(Q_{\phi_i}(s,a)-y)^2],\qquad
  J_\pi=\mathbb E[\alpha\log\pi_\theta(a\mid s)-\min_iQ_{\phi_i}(s,a)].$$

Here $z$ means **true MDP termination**. A Gymnasium `TimeLimit` truncation
resets interaction but does not erase the bootstrap term. Temperature is
learned with target entropy $-\dim(\mathcal A)=-2$.

References: [Gymnasium Lunar Lander](https://gymnasium.farama.org/environments/box2d/lunar_lander/),
[SAC algorithms and applications](https://arxiv.org/abs/1812.05905), and the
readable [CleanRL SAC implementation](https://github.com/vwxyzjn/cleanrl/blob/master/cleanrl/sac_continuous_action.py).


In [ ]:
class ReplayBuffer:
    def __init__(self, observation_dim, action_dim, capacity, seed):
        self.observations = np.empty((capacity, observation_dim), dtype=np.float32)
        self.actions = np.empty((capacity, action_dim), dtype=np.float32)
        self.rewards = np.empty((capacity, 1), dtype=np.float32)
        self.next_observations = np.empty((capacity, observation_dim), dtype=np.float32)
        self.terminated = np.empty((capacity, 1), dtype=np.float32)
        self.capacity = capacity
        self.position = 0
        self.size = 0
        self.rng = np.random.default_rng(seed)

    def add(self, observation, action, reward, next_observation, terminated):
        index = self.position
        self.observations[index] = observation
        self.actions[index] = action
        self.rewards[index, 0] = reward
        self.next_observations[index] = next_observation
        self.terminated[index, 0] = terminated
        self.position = (self.position + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size, device):
        if self.size < batch_size:
            raise ValueError("Not enough transitions for one batch")
        indices = self.rng.integers(0, self.size, size=batch_size)
        arrays = (
            self.observations[indices],
            self.actions[indices],
            self.rewards[indices],
            self.next_observations[indices],
            self.terminated[indices],
        )
        return tuple(torch.as_tensor(array, device=device) for array in arrays)

    def __len__(self):
        return self.size


replay_probe = ReplayBuffer(OBS_DIM, ACT_DIM, config.batch_size, config.seed)
for _ in range(config.batch_size):
    replay_probe.add(
        np.zeros(OBS_DIM, dtype=np.float32),
        np.zeros(ACT_DIM, dtype=np.float32),
        0.0,
        np.ones(OBS_DIM, dtype=np.float32),
        False,
    )
probe_batch = replay_probe.sample(config.batch_size, DEVICE)
assert probe_batch[0].shape == (config.batch_size, OBS_DIM)
assert probe_batch[1].shape == (config.batch_size, ACT_DIM)
del replay_probe, probe_batch


## 2. Bounded stochastic actor and twin critics

Let $u=\mu_\theta(s)+\sigma_\theta(s)\epsilon$ and transform it to arbitrary
Box bounds with $a=c+d\odot\tanh u$. The action density must include the
affine and tanh Jacobians. The implementation uses the stable identity

$$\log(1-\tanh^2u)=2(\log2-u-\operatorname{softplus}(-2u)).$$


In [ ]:
LOG_STD_MIN, LOG_STD_MAX = -5.0, 2.0


def build_mlp(input_dim, hidden_sizes, output_dim):
    layers = []
    previous = input_dim
    for width in hidden_sizes:
        layer = nn.Linear(previous, width)
        nn.init.xavier_uniform_(layer.weight)
        nn.init.zeros_(layer.bias)
        layers.extend((layer, nn.ReLU()))
        previous = width
    output = nn.Linear(previous, output_dim)
    nn.init.uniform_(output.weight, -3e-3, 3e-3)
    nn.init.zeros_(output.bias)
    layers.append(output)
    return nn.Sequential(*layers)


class SquashedGaussianActor(nn.Module):
    def __init__(self, observation_dim, action_low, action_high, hidden_sizes):
        super().__init__()
        self.body = build_mlp(observation_dim, hidden_sizes, hidden_sizes[-1])[:-1]
        self.mean = nn.Linear(hidden_sizes[-1], len(action_low))
        self.log_std = nn.Linear(hidden_sizes[-1], len(action_low))
        nn.init.uniform_(self.mean.weight, -3e-3, 3e-3)
        nn.init.uniform_(self.log_std.weight, -3e-3, 3e-3)
        nn.init.zeros_(self.mean.bias)
        nn.init.zeros_(self.log_std.bias)
        action_low = torch.as_tensor(action_low, dtype=torch.float32)
        action_high = torch.as_tensor(action_high, dtype=torch.float32)
        self.register_buffer("action_scale", (action_high - action_low) / 2)
        self.register_buffer("action_bias", (action_high + action_low) / 2)

    def distribution_parameters(self, observation):
        hidden = self.body(observation)
        mean = self.mean(hidden)
        raw_log_std = torch.tanh(self.log_std(hidden))
        log_std = LOG_STD_MIN + 0.5 * (LOG_STD_MAX - LOG_STD_MIN) * (raw_log_std + 1)
        return mean, log_std

    def sample(self, observation):
        mean, log_std = self.distribution_parameters(observation)
        normal = Normal(mean, log_std.exp())
        raw_action = normal.rsample()
        squashed = torch.tanh(raw_action)
        action = self.action_bias + self.action_scale * squashed
        log_tanh_jacobian = 2 * (
            np.log(2.0) - raw_action - F.softplus(-2 * raw_action)
        )
        log_probability = (
            normal.log_prob(raw_action)
            - log_tanh_jacobian
            - torch.log(self.action_scale)
        ).sum(dim=-1, keepdim=True)
        return action, log_probability, log_std

    def deterministic(self, observation):
        mean, _ = self.distribution_parameters(observation)
        return self.action_bias + self.action_scale * torch.tanh(mean)


class Critic(nn.Module):
    def __init__(self, observation_dim, action_dim, hidden_sizes):
        super().__init__()
        self.network = build_mlp(observation_dim + action_dim, hidden_sizes, 1)

    def forward(self, observation, action):
        return self.network(torch.cat((observation, action), dim=-1))


actor_probe = SquashedGaussianActor(
    OBS_DIM, ACTION_LOW, ACTION_HIGH, config.hidden_sizes
).to(DEVICE)
probe_states = torch.zeros((16, OBS_DIM), device=DEVICE)
probe_actions, probe_log_probabilities, _ = actor_probe.sample(probe_states)
assert torch.isfinite(probe_log_probabilities).all()
assert torch.all(probe_actions >= torch.as_tensor(ACTION_LOW, device=DEVICE) - 1e-6)
assert torch.all(probe_actions <= torch.as_tensor(ACTION_HIGH, device=DEVICE) + 1e-6)
del actor_probe, probe_states, probe_actions, probe_log_probabilities


In [ ]:
def soft_update(target, source, tau):
    with torch.no_grad():
        for target_parameter, source_parameter in zip(
            target.parameters(), source.parameters(), strict=True
        ):
            target_parameter.mul_(1 - tau).add_(source_parameter, alpha=tau)


class SACAgent:
    def __init__(self, config, device):
        self.config = config
        self.device = device
        self.actor = SquashedGaussianActor(
            OBS_DIM, ACTION_LOW, ACTION_HIGH, config.hidden_sizes
        ).to(device)
        self.q1 = Critic(OBS_DIM, ACT_DIM, config.hidden_sizes).to(device)
        self.q2 = Critic(OBS_DIM, ACT_DIM, config.hidden_sizes).to(device)
        self.q1_target = copy.deepcopy(self.q1).requires_grad_(False)
        self.q2_target = copy.deepcopy(self.q2).requires_grad_(False)
        self.actor_optimizer = torch.optim.Adam(
            self.actor.parameters(), lr=config.actor_lr
        )
        self.critic_parameters = list(self.q1.parameters()) + list(self.q2.parameters())
        self.critic_optimizer = torch.optim.Adam(
            self.critic_parameters, lr=config.critic_lr
        )
        self.log_alpha = torch.tensor(
            np.log(config.initial_alpha),
            dtype=torch.float32,
            device=device,
            requires_grad=True,
        )
        self.alpha_optimizer = torch.optim.Adam(
            [self.log_alpha], lr=config.alpha_lr
        )
        self.target_entropy = -float(ACT_DIM)

    @property
    def alpha(self):
        return self.log_alpha.exp()

    def act(self, observation, deterministic=False):
        state = torch.as_tensor(observation, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            if deterministic:
                action = self.actor.deterministic(state)
            else:
                action, _, _ = self.actor.sample(state)
        return action.cpu().numpy().astype(np.float32)

    def update(self, batch):
        observations, actions, rewards, next_observations, terminated = batch
        with torch.no_grad():
            next_actions, next_log_probabilities, _ = self.actor.sample(next_observations)
            next_soft_values = torch.minimum(
                self.q1_target(next_observations, next_actions),
                self.q2_target(next_observations, next_actions),
            ) - self.alpha.detach() * next_log_probabilities
            targets = rewards + self.config.gamma * (1 - terminated) * next_soft_values

        q1_values = self.q1(observations, actions)
        q2_values = self.q2(observations, actions)
        q1_loss = F.mse_loss(q1_values, targets)
        q2_loss = F.mse_loss(q2_values, targets)
        critic_loss = q1_loss + q2_loss
        self.critic_optimizer.zero_grad(set_to_none=True)
        critic_loss.backward()
        critic_gradient_norm = torch.nn.utils.clip_grad_norm_(
            self.critic_parameters, self.config.max_grad_norm
        )
        self.critic_optimizer.step()

        for parameter in self.critic_parameters:
            parameter.requires_grad_(False)
        sampled_actions, log_probabilities, log_std = self.actor.sample(observations)
        sampled_q = torch.minimum(
            self.q1(observations, sampled_actions),
            self.q2(observations, sampled_actions),
        )
        actor_loss = (
            self.alpha.detach() * log_probabilities - sampled_q
        ).mean()
        self.actor_optimizer.zero_grad(set_to_none=True)
        actor_loss.backward()
        actor_gradient_norm = torch.nn.utils.clip_grad_norm_(
            self.actor.parameters(), self.config.max_grad_norm
        )
        self.actor_optimizer.step()
        for parameter in self.critic_parameters:
            parameter.requires_grad_(True)

        alpha_loss = -(
            self.alpha * (log_probabilities.detach() + self.target_entropy)
        ).mean()
        self.alpha_optimizer.zero_grad(set_to_none=True)
        alpha_loss.backward()
        self.alpha_optimizer.step()

        soft_update(self.q1_target, self.q1, self.config.tau)
        soft_update(self.q2_target, self.q2, self.config.tau)

        absolute_td_error = 0.5 * (
            (q1_values.detach() - targets).abs()
            + (q2_values.detach() - targets).abs()
        )
        return {
            "q1_loss": float(q1_loss.detach()),
            "q2_loss": float(q2_loss.detach()),
            "actor_loss": float(actor_loss.detach()),
            "alpha_loss": float(alpha_loss.detach()),
            "alpha": float(self.alpha.detach()),
            "entropy": float(-log_probabilities.detach().mean()),
            "mean_log_std": float(log_std.detach().mean()),
            "mean_q": float(0.5 * (q1_values.detach() + q2_values.detach()).mean()),
            "mean_target": float(targets.mean()),
            "q_disagreement": float((q1_values.detach() - q2_values.detach()).abs().mean()),
            "mean_absolute_td_error": float(absolute_td_error.mean()),
            "actor_gradient_norm": float(actor_gradient_norm),
            "critic_gradient_norm": float(critic_gradient_norm),
        }


# A truncation must retain the bootstrap; a true termination must not.
reward_probe = torch.tensor([[1.0]])
bootstrap_probe = torch.tensor([[7.0]])
terminated_targets = reward_probe + config.gamma * (1 - torch.tensor([[1.0]])) * bootstrap_probe
truncated_targets = reward_probe + config.gamma * (1 - torch.tensor([[0.0]])) * bootstrap_probe
assert float(terminated_targets) == 1.0
assert float(truncated_targets) > 1.0


## 3. Training and paired-seed evaluation

Training uses exploratory actions, while evaluation uses the deterministic
squashed mean on the same held-out seeds at every checkpoint. This common
random-number design makes changes over training easier to interpret.
Evaluation does not choose checkpoints; a separate final seed panel should be
used for publication claims.

The full profile follows the scale of RL Zoo's current LunarLanderContinuous
SAC baseline (500k steps, 10k random warm-up, batch 256, automatic entropy),
while keeping a constant conservative learning rate. See the
[registered comparison configuration](https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/sac.yml).


In [ ]:
def make_lander(render_mode=None):
    return gym.make(
        config.env_id,
        continuous=True,
        enable_wind=False,
        render_mode=render_mode,
    )


def evaluate_policy(agent, seeds):
    rows = []
    environment = make_lander()
    try:
        for seed in seeds:
            observation, _ = environment.reset(seed=int(seed))
            episode_return = 0.0
            episode_length = 0
            main_engine_steps = 0
            lateral_engine_steps = 0
            saturated_components = 0
            terminated = truncated = False
            while not (terminated or truncated):
                action = agent.act(observation, deterministic=True)
                observation, reward, terminated, truncated, _ = environment.step(action)
                episode_return += float(reward)
                episode_length += 1
                main_engine_steps += int(action[0] > 0)
                lateral_engine_steps += int(abs(action[1]) > 0.5)
                saturated_components += int(np.sum(np.abs(action) > 0.95))
            rows.append(
                {
                    "seed": int(seed),
                    "episode_return": episode_return,
                    "episode_length": episode_length,
                    "solved": episode_return >= 200,
                    "terminated": bool(terminated),
                    "truncated": bool(truncated),
                    "main_engine_rate": main_engine_steps / episode_length,
                    "lateral_engine_rate": lateral_engine_steps / episode_length,
                    "action_saturation_rate": saturated_components / (ACT_DIM * episode_length),
                }
            )
    finally:
        environment.close()
    return pd.DataFrame(rows)


@dataclass
class TrainingResult:
    agent: SACAgent
    episodes: pd.DataFrame
    updates: pd.DataFrame
    evaluations: pd.DataFrame
    elapsed_seconds: float


EVALUATION_SEEDS = tuple(
    range(10_000, 10_000 + config.eval_episodes)
)


def train_sac(config):
    seed_everything(config.seed)
    environment = make_lander()
    environment.action_space.seed(config.seed)
    agent = SACAgent(config, DEVICE)
    replay = ReplayBuffer(
        OBS_DIM,
        ACT_DIM,
        config.replay_capacity,
        seed=config.seed + 1,
    )
    episode_rows, update_rows, evaluation_frames = [], [], []
    episode_index = 0
    episode_return = 0.0
    episode_length = 0
    episode_main_steps = 0
    episode_lateral_steps = 0
    observation, _ = environment.reset(seed=config.seed)
    start_time = time.perf_counter()

    initial_evaluation = evaluate_policy(agent, EVALUATION_SEEDS)
    initial_evaluation.insert(0, "step", 0)
    evaluation_frames.append(initial_evaluation)

    try:
        for step in range(1, config.total_steps + 1):
            if step <= config.learning_starts:
                action = environment.action_space.sample()
            else:
                action = agent.act(observation)

            next_observation, reward, terminated, truncated, _ = environment.step(action)
            replay.add(observation, action, reward, next_observation, terminated)
            episode_return += float(reward)
            episode_length += 1
            episode_main_steps += int(action[0] > 0)
            episode_lateral_steps += int(abs(action[1]) > 0.5)
            observation = next_observation

            if step > config.learning_starts and len(replay) >= config.batch_size:
                diagnostics = agent.update(replay.sample(config.batch_size, DEVICE))
                if step % config.diagnostic_interval == 0:
                    update_rows.append({"step": step, **diagnostics})

            if terminated or truncated:
                episode_index += 1
                episode_rows.append(
                    {
                        "step": step,
                        "episode": episode_index,
                        "episode_return": episode_return,
                        "episode_length": episode_length,
                        "solved": episode_return >= 200,
                        "terminated": bool(terminated),
                        "truncated": bool(truncated),
                        "main_engine_rate": episode_main_steps / episode_length,
                        "lateral_engine_rate": episode_lateral_steps / episode_length,
                    }
                )
                observation, _ = environment.reset()
                episode_return = 0.0
                episode_length = 0
                episode_main_steps = 0
                episode_lateral_steps = 0

            if step % config.eval_interval == 0 or step == config.total_steps:
                evaluation = evaluate_policy(agent, EVALUATION_SEEDS)
                evaluation.insert(0, "step", step)
                evaluation_frames.append(evaluation)
                mean_return = evaluation["episode_return"].mean()
                print(
                    f"step={step:>7,}  episodes={episode_index:>4}  "
                    f"eval_return={mean_return:>8.1f}  "
                    f"alpha={float(agent.alpha.detach()):.3f}"
                )
    finally:
        environment.close()

    return TrainingResult(
        agent=agent,
        episodes=pd.DataFrame(episode_rows),
        updates=pd.DataFrame(update_rows),
        evaluations=pd.concat(evaluation_frames, ignore_index=True),
        elapsed_seconds=time.perf_counter() - start_time,
    )


In [ ]:
result = train_sac(config)
print(
    f"finished {config.total_steps:,} steps and {len(result.episodes)} complete episodes "
    f"in {result.elapsed_seconds:.1f}s"
)


## 4. Diagnostic dashboard

A return curve alone cannot distinguish weak exploration from critic failure.
We retain deterministic evaluation distributions, TD scale, twin-critic
disagreement, entropy temperature, policy spread, and Lunar-specific engine
activation. The official environment calls an episode solved at return 200;
the line is a task convention, not a confidence interval.


In [ ]:
episodes = result.episodes.copy()
updates = result.updates.copy()
evaluations = result.evaluations.copy()
evaluation_summary = (
    evaluations.groupby("step", as_index=False)
    .agg(
        mean_return=("episode_return", "mean"),
        return_std=("episode_return", "std"),
        solved_fraction=("solved", "mean"),
        mean_length=("episode_length", "mean"),
    )
    .fillna({"return_std": 0.0})
)

fig, axes = plt.subplots(2, 3, figsize=(15, 8.2))
if not episodes.empty:
    axes[0, 0].plot(episodes["step"], episodes["episode_return"], alpha=0.35)
    axes[0, 0].plot(
        episodes["step"],
        episodes["episode_return"].rolling(20, min_periods=1).mean(),
        linewidth=2,
        label="20-episode mean",
    )
axes[0, 0].axhline(200, color="tab:green", linestyle="--", label="solved threshold")
axes[0, 0].set(title="Exploratory training return", xlabel="environment step")
axes[0, 0].legend()

x = evaluation_summary["step"].to_numpy()
mean_return = evaluation_summary["mean_return"].to_numpy()
return_std = evaluation_summary["return_std"].to_numpy()
axes[0, 1].plot(x, mean_return, marker="o", label="deterministic mean")
axes[0, 1].fill_between(x, mean_return - return_std, mean_return + return_std, alpha=0.2)
axes[0, 1].axhline(200, color="tab:green", linestyle="--")
axes[0, 1].set(title="Paired-seed evaluation", xlabel="environment step")
solved_axis = axes[0, 1].twinx()
solved_axis.plot(x, evaluation_summary["solved_fraction"], color="tab:green", alpha=0.6)
solved_axis.set(ylabel="solved fraction", ylim=(-0.03, 1.03))

if not updates.empty:
    axes[0, 2].plot(updates["step"], updates["q1_loss"], label="Q1")
    axes[0, 2].plot(updates["step"], updates["q2_loss"], label="Q2", alpha=0.75)
    axes[0, 2].set_yscale("log")
    axes[0, 2].legend()
    axes[1, 0].plot(
        updates["step"], updates["mean_absolute_td_error"], label="|TD error|"
    )
    axes[1, 0].plot(
        updates["step"], updates["q_disagreement"], label="|Q1-Q2|", alpha=0.8
    )
    axes[1, 0].set_yscale("log")
    axes[1, 0].legend()
    axes[1, 1].plot(updates["step"], updates["actor_loss"], label="actor loss")
    alpha_axis = axes[1, 1].twinx()
    alpha_axis.plot(updates["step"], updates["alpha"], color="tab:red", label="alpha")
    alpha_axis.set_ylabel("temperature alpha", color="tab:red")
    axes[1, 2].plot(updates["step"], updates["entropy"], label="policy entropy")
    axes[1, 2].plot(updates["step"], updates["mean_log_std"], label="mean log std")
    axes[1, 2].legend()
axes[0, 2].set(title="Twin critic losses", xlabel="environment step")
axes[1, 0].set(title="Critic uncertainty proxies", xlabel="environment step")
axes[1, 1].set(title="Actor objective / temperature", xlabel="environment step")
axes[1, 2].set(title="Exploration diagnostics", xlabel="environment step")
plt.tight_layout()
if SMOKE:
    plt.close(fig)
else:
    plt.show()

action_summary = (
    evaluations.groupby("step", as_index=False)
    .agg(
        main_engine_rate=("main_engine_rate", "mean"),
        lateral_engine_rate=("lateral_engine_rate", "mean"),
        action_saturation_rate=("action_saturation_rate", "mean"),
    )
)
engine_figure, engine_axes = plt.subplots(1, 2, figsize=(13, 4))
if not episodes.empty:
    engine_axes[0].plot(
        episodes["step"],
        episodes["main_engine_rate"].rolling(20, min_periods=1).mean(),
        label="main engine",
    )
    engine_axes[0].plot(
        episodes["step"],
        episodes["lateral_engine_rate"].rolling(20, min_periods=1).mean(),
        label="lateral engines",
    )
engine_axes[0].set(
    xlabel="environment step",
    ylabel="activation rate",
    ylim=(-0.03, 1.03),
    title="Exploratory engine use",
)
engine_axes[0].legend()
engine_axes[1].plot(
    action_summary["step"], action_summary["main_engine_rate"], marker="o", label="main"
)
engine_axes[1].plot(
    action_summary["step"], action_summary["lateral_engine_rate"], marker="o", label="lateral"
)
engine_axes[1].plot(
    action_summary["step"],
    action_summary["action_saturation_rate"],
    marker="o",
    label="saturated components",
)
engine_axes[1].set(
    xlabel="environment step",
    ylabel="rate",
    ylim=(-0.03, 1.03),
    title="Deterministic evaluation actions",
)
engine_axes[1].legend()
plt.tight_layout()
if SMOKE:
    plt.close(engine_figure)
else:
    plt.show()

final_step = evaluations["step"].max()
final_evaluation = evaluations[evaluations["step"] == final_step]
final_summary = pd.Series(
    {
        "training_step": int(final_step),
        "mean_return": final_evaluation["episode_return"].mean(),
        "return_std": final_evaluation["episode_return"].std(ddof=0),
        "median_return": final_evaluation["episode_return"].median(),
        "solved_fraction": final_evaluation["solved"].mean(),
        "mean_episode_length": final_evaluation["episode_length"].mean(),
        "main_engine_rate": final_evaluation["main_engine_rate"].mean(),
        "lateral_engine_rate": final_evaluation["lateral_engine_rate"].mean(),
        "action_saturation_rate": final_evaluation["action_saturation_rate"].mean(),
    },
    name="final deterministic evaluation",
)
display(final_summary.to_frame())


## 5. Pick a showcase without contaminating the benchmark

The animation seeds are disjoint from the paired evaluation seeds. If at
least one fixed showcase seed is solved, we animate the median solved rollout;
otherwise we show the best available attempt and label it honestly. Only the
seed-aggregated panel above is performance evidence.


In [ ]:
@dataclass
class Rollout:
    seed: int
    total_return: float
    terminated: bool
    truncated: bool
    observations: np.ndarray
    actions: np.ndarray
    rewards: np.ndarray
    frames: list[np.ndarray]
    frame_steps: np.ndarray


def record_rollout(agent, seed, capture_stride=2):
    environment = make_lander(render_mode="rgb_array")
    observations, actions, rewards, frames, frame_steps = [], [], [], [], []
    try:
        observation, _ = environment.reset(seed=int(seed))
        observations.append(observation.copy())
        frames.append(environment.render()[::2, ::2].copy())
        frame_steps.append(0)
        terminated = truncated = False
        step = 0
        while not (terminated or truncated):
            action = agent.act(observation, deterministic=True)
            observation, reward, terminated, truncated, _ = environment.step(action)
            step += 1
            actions.append(action.copy())
            rewards.append(float(reward))
            observations.append(observation.copy())
            if step % capture_stride == 0 or terminated or truncated:
                frames.append(environment.render()[::2, ::2].copy())
                frame_steps.append(step)
    finally:
        environment.close()
    return Rollout(
        seed=int(seed),
        total_return=float(np.sum(rewards)),
        terminated=bool(terminated),
        truncated=bool(truncated),
        observations=np.asarray(observations, dtype=np.float32),
        actions=np.asarray(actions, dtype=np.float32),
        rewards=np.asarray(rewards, dtype=np.float32),
        frames=frames,
        frame_steps=np.asarray(frame_steps, dtype=int),
    )


showcase_count = 1 if SMOKE else (4 if QUICK else 12)
SHOWCASE_SEEDS = tuple(range(20_000, 20_000 + showcase_count))
showcase_scores = evaluate_policy(result.agent, SHOWCASE_SEEDS)
solved_scores = showcase_scores[showcase_scores["solved"]]
if not solved_scores.empty:
    solved_median = solved_scores["episode_return"].median()
    selected_row = solved_scores.iloc[
        (solved_scores["episode_return"] - solved_median).abs().argmin()
    ]
    selection_reason = "median solved showcase"
else:
    selected_row = showcase_scores.loc[showcase_scores["episode_return"].idxmax()]
    selection_reason = "best available attempt; no showcase seed solved"

showcase = record_rollout(result.agent, int(selected_row["seed"]))
showcase_scores = showcase_scores.assign(
    selected=showcase_scores["seed"].eq(showcase.seed)
)
print(f"Selected seed {showcase.seed}: {selection_reason}")
display(showcase_scores)


## 6. Landing replay with flight telemetry

The animation is generated from a fresh deterministic evaluation episode.
Its heads-up display reports position, velocity, attitude, engine commands,
and accumulated reward. Playback is downsampled to at most 300 frames so the
notebook remains responsive.


In [ ]:
def make_landing_animation(rollout, fps=30, max_frames=300):
    if len(rollout.frames) > max_frames:
        frame_indices = np.linspace(
            0, len(rollout.frames) - 1, max_frames, dtype=int
        )
    else:
        frame_indices = np.arange(len(rollout.frames))
    cumulative_rewards = np.cumsum(rollout.rewards)

    fig, axis = plt.subplots(figsize=(8, 5.4), dpi=80)
    fig.patch.set_facecolor("#070b14")
    axis.set_facecolor("#070b14")
    image = axis.imshow(rollout.frames[int(frame_indices[0])])
    axis.set_axis_off()
    status = "SOLVED" if rollout.total_return >= 200 else "ATTEMPT"
    title = axis.set_title(
        f"LunarLander SAC — {status} — seed {rollout.seed}",
        color="white",
        fontsize=14,
        pad=10,
    )
    hud = axis.text(
        0.018,
        0.975,
        "",
        transform=axis.transAxes,
        va="top",
        ha="left",
        color="#e8f1ff",
        family="monospace",
        fontsize=9,
        bbox={"boxstyle": "round,pad=0.55", "facecolor": "#08101f", "alpha": 0.82, "edgecolor": "#4ea5ff"},
    )

    def update(animation_index):
        stored_index = int(frame_indices[animation_index])
        step = int(rollout.frame_steps[stored_index])
        state = rollout.observations[min(step, len(rollout.observations) - 1)]
        if step:
            action = rollout.actions[min(step - 1, len(rollout.actions) - 1)]
            accumulated_reward = cumulative_rewards[min(step - 1, len(cumulative_rewards) - 1)]
        else:
            action = np.zeros(ACT_DIM)
            accumulated_reward = 0.0
        image.set_data(rollout.frames[stored_index])
        hud.set_text(
            f"step      {step:4d}\n"
            f"return   {accumulated_reward:7.1f} / {rollout.total_return:7.1f}\n"
            f"x, y     {state[0]:+6.3f}  {state[1]:+6.3f}\n"
            f"vx, vy   {state[2]:+6.3f}  {state[3]:+6.3f}\n"
            f"angle    {state[4]:+6.3f}\n"
            f"main     {action[0]:+6.3f}\n"
            f"lateral  {action[1]:+6.3f}"
        )
        return image, hud, title

    movie = animation.FuncAnimation(
        fig,
        update,
        frames=len(frame_indices),
        interval=1_000 / fps,
        blit=True,
        repeat=False,
    )
    return fig, movie


if SMOKE:
    smoke_figure = plt.figure(figsize=(6, 4))
    plt.imshow(showcase.frames[-1])
    plt.axis("off")
    plt.title("Animation HTML skipped only in automated smoke mode")
    plt.close(smoke_figure)
    landing_movie = None
else:
    plt.rcParams["animation.embed_limit"] = 120
    landing_figure, landing_movie = make_landing_animation(showcase)
    landing_html = HTML(landing_movie.to_jshtml(fps=30, default_mode="once"))
    plt.close(landing_figure)
    display(landing_html)

# Optional portable artifact. Toggle after training if you want a GIF on disk.
SAVE_GIF = False
if SAVE_GIF and landing_movie is not None:
    output_directory = Path("results/lunar_lander_sac")
    output_directory.mkdir(parents=True, exist_ok=True)
    gif_path = output_directory / f"landing_seed_{showcase.seed}.gif"
    landing_movie.save(gif_path, writer=animation.PillowWriter(fps=30))
    print(f"saved {gif_path}")


In [ ]:
states = showcase.observations
actions = showcase.actions
state_steps = np.arange(len(states))
action_steps = np.arange(1, len(actions) + 1)
cumulative_reward = np.cumsum(showcase.rewards)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0, 0].plot(states[:, 0], states[:, 1], color="tab:blue")
axes[0, 0].scatter(states[0, 0], states[0, 1], marker="o", label="start")
axes[0, 0].scatter(states[-1, 0], states[-1, 1], marker="X", s=80, label="finish")
axes[0, 0].scatter(0, 0, marker="*", s=110, color="tab:green", label="pad center")
axes[0, 0].set(xlabel="x", ylabel="y", title="Flight path in observation coordinates")
axes[0, 0].legend()

axes[0, 1].plot(state_steps, states[:, 2], label="horizontal velocity")
axes[0, 1].plot(state_steps, states[:, 3], label="vertical velocity")
axes[0, 1].plot(state_steps, states[:, 4], label="angle", alpha=0.8)
axes[0, 1].axhline(0, color="black", linewidth=0.8)
axes[0, 1].set(xlabel="step", title="Approach stability")
axes[0, 1].legend()

axes[1, 0].plot(action_steps, actions[:, 0], label="main")
axes[1, 0].plot(action_steps, actions[:, 1], label="lateral")
axes[1, 0].axhline(0, color="tab:blue", linestyle="--", alpha=0.5)
axes[1, 0].axhline(0.5, color="tab:orange", linestyle=":", alpha=0.5)
axes[1, 0].axhline(-0.5, color="tab:orange", linestyle=":", alpha=0.5)
axes[1, 0].set(xlabel="step", ylabel="action", title="Engine commands and dead zones")
axes[1, 0].legend()

axes[1, 1].plot(action_steps, cumulative_reward, color="tab:green")
axes[1, 1].set(xlabel="step", ylabel="cumulative reward", title="Reward accumulation")
plt.tight_layout()
if SMOKE:
    plt.close(fig)
else:
    plt.show()


## 7. What to trust, save, and try next

- `QUICK=True` validates mechanics. It is expected to crash often.
- For a real baseline, use `QUICK=False`, at least three independent training
  seeds, and 50+ untouched final evaluation seeds. Report the distribution,
  not the best animation.
- Wind is disabled for the baseline. Turn it on only as a separately labeled
  robustness experiment.
- SAC can still fail through critic extrapolation, temperature collapse,
  overconfident Q targets, or action saturation; the dashboard is designed to
  make those failures visible.

The trained actor remains in `result.agent`. Set `SAVE_GIF=True` to export the
replay, or save `result.agent.actor.state_dict()` together with `asdict(config)`
when you want a checkpoint.
